# ⚡ Electricity Load Forecasting — Decision Tree & Random Forest
**Dataset:** `continuous_dataset.csv` (Panama, 2015–2020, hourly)  
**Course:** UCL BENV0148 Advanced Machine Learning

---

### 🎯 Forecasting Goals
| | |
|---|---|
| **Primary horizon** | **12-hour ahead** — predict next 12 h using 24 h lookback |
| **Secondary horizon** | **24-hour ahead** — predict full next day |
| **Output strategy** | **Direct Multi-Output** — all steps predicted simultaneously |
| **Models compared** | Decision Tree vs Random Forest (both horizons) |

---

### 📋 Pipeline
```
EDA ──► Preprocessing ──► Feature Engineering ──► CV Training ──► Evaluation ──► Analysis
                                                        ▲                │
                                                        └── Hyperparameter Tuning ◄─┘
```

| Step | Content |
|---|---|
| 1 | EDA — time series overview, seasonality, distribution, outliers |
| 2 | Data Preprocessing |
| 3 | Feature Engineering (lag, rolling, cyclical, exogenous) |
| 4 | 5-Fold Year-Based Expanding Window CV |
| 5 | Hyperparameter Tuning (GridSearchCV on CV1) |
| 6 | Model Training — Decision Tree & Random Forest (12h & 24h) |
| 7 | Model Evaluation — MAE / RMSE / MAPE across folds |
| 8 | Visualisation — Actual vs Predicted |
| 9 | Feature Importance Analysis |
| 10 | Robustness & Overfitting Analysis |


## 🔧 Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import wilcoxon

from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
SEED = 42
np.random.seed(SEED)

# ── Forecast constants ────────────────────────────────────────────────────────
HORIZON_12    = 12     # primary:   12-hour ahead forecast
HORIZON_24    = 24     # secondary: 24-hour ahead forecast
HISTORY_HOURS = 24     # lookback window
TARGET_COL    = 'nat_demand'

print('✅ All imports OK')
print(f'   Primary   : {HORIZON_12}-step  ({HORIZON_12}h ahead)')
print(f'   Secondary : {HORIZON_24}-step  ({HORIZON_24}h ahead)')
print(f'   Lookback  : {HISTORY_HOURS}h')

## 📂 Load Dataset

In [ ]:
# ── Option A: Google Colab upload ─────────────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = list(uploaded.keys())[0]

# ── Option B: local path ──────────────────────────────────────────────────────
DATA_PATH = 'continuous_dataset.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f'Shape      : {df.shape}')
print(f'Date range : {df.datetime.min()}  →  {df.datetime.max()}')
print(f'Columns    : {df.columns.tolist()}')
df.head(3)

## Step 1: EDA — Exploratory Data Analysis

In [ ]:
# ── 1a. Basic stats & missing values ─────────────────────────────────────────
display(df.describe().round(3))

missing = df.isnull().sum()
print('\nMissing values per column:')
print(missing[missing > 0] if missing.any() else 'None ✅')

# Timestamp continuity check
df_time = df.set_index('datetime')
full_range = pd.date_range(df_time.index.min(), df_time.index.max(), freq='h')
missing_ts = full_range.difference(df_time.index)
print(f'Missing timestamps: {len(missing_ts)}')

In [ ]:
# ── 1b. Full time-series overview ────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

axes[0].plot(df['datetime'], df[TARGET_COL], lw=0.5, color='steelblue', alpha=0.8)
axes[0].set_title('National Electricity Demand — Full Series (2015–2020)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Demand (MW)')
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[0].grid(True, alpha=0.3)

sample_week = df[(df['datetime'] >= '2017-01-02') & (df['datetime'] < '2017-01-09')]
axes[1].plot(sample_week['datetime'], sample_week[TARGET_COL],
             lw=1.5, color='darkorange', marker='o', markersize=2)
axes[1].set_title('Zoom: One Week (Jan 2017) — Hourly Pattern', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Demand (MW)')
axes[1].grid(True, alpha=0.3)

axes[2].hist(df[TARGET_COL], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[2].set_title('Distribution of nat_demand', fontsize=13, fontweight='bold')
axes[2].set_xlabel('MW')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 1c. Seasonality patterns ──────────────────────────────────────────────────
tmp = df.copy()
tmp['hour']      = tmp['datetime'].dt.hour
tmp['dayofweek'] = tmp['datetime'].dt.dayofweek
tmp['month']     = tmp['datetime'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

tmp.groupby('hour')[TARGET_COL].mean().plot(ax=axes[0], color='steelblue', marker='o', ms=4)
axes[0].set_title('Average Demand by Hour of Day')
axes[0].set_xlabel('Hour');  axes[0].set_ylabel('MW')
axes[0].grid(True, alpha=0.3)

day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
tmp.groupby('dayofweek')[TARGET_COL].mean().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_xticklabels(day_labels, rotation=0)
axes[1].set_title('Average Demand by Day of Week')
axes[1].set_ylabel('MW')
axes[1].grid(True, alpha=0.3, axis='y')

tmp.groupby('month')[TARGET_COL].mean().plot(kind='bar', ax=axes[2], color='seagreen', edgecolor='white')
axes[2].set_title('Average Demand by Month')
axes[2].set_xlabel('Month');  axes[2].set_ylabel('MW')
axes[2].set_xticklabels(range(1,13), rotation=0)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── 1d. Outlier detection ─────────────────────────────────────────────────────
Q1, Q3 = df[TARGET_COL].quantile(0.25), df[TARGET_COL].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df[(df[TARGET_COL] < lower_fence) | (df[TARGET_COL] > upper_fence)]
print(f'IQR fences: [{lower_fence:.1f}, {upper_fence:.1f}] MW')
print(f'Outliers detected: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')
display(outliers[['datetime', TARGET_COL]].sort_values(TARGET_COL).head(10))

## Step 2: Data Preprocessing

In [ ]:
df = df.sort_values('datetime').reset_index(drop=True)
df = df.set_index('datetime').asfreq('h')

# Forward-fill any gaps (from asfreq)
df[TARGET_COL] = df[TARGET_COL].fillna(method='ffill').fillna(method='bfill')

# Restrict to clean range used by other group members
df = df.loc['2015-01-01':'2020-02-01']

print(f'Preprocessed shape : {df.shape}')
print(f'Date range         : {df.index.min()} → {df.index.max()}')
print(f'Missing values     : {df[TARGET_COL].isna().sum()}')

## Step 3: Feature Engineering

In [ ]:
def add_time_features(data):
    """Cyclical time encoding + calendar flags"""
    d = data.copy()
    d['hour']        = d.index.hour
    d['dayofweek']   = d.index.dayofweek
    d['month']       = d.index.month
    d['is_weekend']  = (d['dayofweek'] >= 5).astype(int)
    d['weekofyear']  = d.index.isocalendar().week.astype(int)
    # Cyclical encoding
    d['hour_sin']    = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos']    = np.cos(2 * np.pi * d['hour'] / 24)
    d['dow_sin']     = np.sin(2 * np.pi * d['dayofweek'] / 7)
    d['dow_cos']     = np.cos(2 * np.pi * d['dayofweek'] / 7)
    d['month_sin']   = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos']   = np.cos(2 * np.pi * d['month'] / 12)
    return d


def add_lag_features(data, target_col, lags):
    """Historical demand at specified lag offsets"""
    d = data.copy()
    for lag in lags:
        d[f'{target_col}_lag_{lag}'] = d[target_col].shift(lag)
    return d


def add_rolling_features(data, target_col, windows):
    """Rolling statistics (shifted by 1 to avoid leakage)"""
    d = data.copy()
    for w in windows:
        base = d[target_col].shift(1)
        d[f'{target_col}_roll_mean_{w}'] = base.rolling(w).mean()
        d[f'{target_col}_roll_std_{w}']  = base.rolling(w).std()
        d[f'{target_col}_roll_min_{w}']  = base.rolling(w).min()
        d[f'{target_col}_roll_max_{w}']  = base.rolling(w).max()
    return d


# ── Apply feature engineering ─────────────────────────────────────────────────
base_lags      = list(range(1, 25))            # 1 h … 24 h
seasonal_lags  = [48, 72, 168]                 # 2d, 3d, 1 week
all_lags       = sorted(set(base_lags + seasonal_lags))
rolling_wins   = [3, 6, 12, 24, 168]

df_feat = add_time_features(df)
df_feat = add_lag_features(df_feat, TARGET_COL, all_lags)
df_feat = add_rolling_features(df_feat, TARGET_COL, rolling_wins)
df_feat = df_feat.dropna()

# Exogenous weather columns present in the dataset
WEATHER_COLS = ['T2M_toc','QV2M_toc','TQL_toc','W2M_toc',
                'T2M_san','QV2M_san','TQL_san','W2M_san',
                'T2M_dav','QV2M_dav','TQL_dav','W2M_dav']
CALENDAR_COLS = ['Holiday_ID','holiday','school']

# All feature columns (exclude target)
FEATURE_COLS = [c for c in df_feat.columns if c != TARGET_COL]

print(f'Feature matrix shape : {df_feat.shape}')
print(f'Number of features   : {len(FEATURE_COLS)}')
print(f'Date range after NaN drop: {df_feat.index.min()} → {df_feat.index.max()}')

## Step 4: Cross-Validation Setup (Expanding Window)

In [ ]:
# ── 5-fold year-based expanding window ───────────────────────────────────────
CV_SPLITS = [
    {'fold': 1, 'train_end': '2015-12-31', 'val_start': '2016-01-01', 'val_end': '2016-12-31'},
    {'fold': 2, 'train_end': '2016-12-31', 'val_start': '2017-01-01', 'val_end': '2017-12-31'},
    {'fold': 3, 'train_end': '2017-12-31', 'val_start': '2018-01-01', 'val_end': '2018-12-31'},
    {'fold': 4, 'train_end': '2018-12-31', 'val_start': '2019-01-01', 'val_end': '2019-12-31'},
    {'fold': 5, 'train_end': '2019-12-31', 'val_start': '2020-01-01', 'val_end': '2020-02-01'},
]

print('CV Splits (Expanding Window):')
print(f'{"Fold":<6} {"Train":<28} {"Validation"}')
print('-' * 60)
for s in CV_SPLITS:
    train_start = '2015-01-01'
    print(f"CV{s['fold']}    {train_start} → {s['train_end']}   {s['val_start']} → {s['val_end']}")

## Step 5: Helper Functions — Sequence Construction & Metrics

In [ ]:
def build_direct_multioutput_dataset(data, feature_cols, target_col, horizon):
    """
    Direct Multi-Output strategy:
    Each sample: features at time t  →  target vector [t+1, t+2, ..., t+horizon]
    No data leakage: lag/rolling features already encode history.
    """
    X_list, Y_list, idx_list = [], [], []
    target_series = data[target_col].values
    feature_matrix = data[feature_cols].values
    n = len(data)

    for i in range(n - horizon):
        X_list.append(feature_matrix[i])
        Y_list.append(target_series[i + 1 : i + 1 + horizon])
        idx_list.append(data.index[i])

    return np.array(X_list), np.array(Y_list), idx_list


def compute_metrics(y_true, y_pred):
    """
    Compute MAE, RMSE, MAPE averaged across all forecast steps.
    y_true, y_pred: shape (n_samples, horizon)
    """
    mae  = mean_absolute_error(y_true.flatten(), y_pred.flatten())
    rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
    # MAPE — guard against zeros
    mask = y_true.flatten() != 0
    mape = np.mean(np.abs((y_true.flatten()[mask] - y_pred.flatten()[mask])
                          / y_true.flatten()[mask])) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}


print('✅ Helper functions defined.')

## Step 6: Hyperparameter Tuning (CV Fold 1, Horizon 12h)

In [ ]:
# ── Use CV1 fold for tuning ───────────────────────────────────────────────────
s = CV_SPLITS[0]
train_data = df_feat.loc['2015-01-01' : s['train_end']]
val_data   = df_feat.loc[s['val_start'] : s['val_end']]

X_tr, Y_tr, _ = build_direct_multioutput_dataset(train_data, FEATURE_COLS, TARGET_COL, HORIZON_12)
X_va, Y_va, _ = build_direct_multioutput_dataset(val_data,   FEATURE_COLS, TARGET_COL, HORIZON_12)

# ── Decision Tree grid ────────────────────────────────────────────────────────
print('🔍 Tuning Decision Tree...')
dt_param_grid = {
    'max_depth'       : [5, 10, 15, 20, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf' : [1, 5, 10],
    'max_features'     : ['sqrt', 'log2', None],
}

best_dt_rmse = np.inf
best_dt_params = {}

for params in ParameterGrid(dt_param_grid):
    dt = MultiOutputRegressor(
        DecisionTreeRegressor(random_state=SEED, **params), n_jobs=-1
    )
    dt.fit(X_tr, Y_tr)
    pred = dt.predict(X_va)
    rmse = np.sqrt(mean_squared_error(Y_va.flatten(), pred.flatten()))
    if rmse < best_dt_rmse:
        best_dt_rmse   = rmse
        best_dt_params = params

print(f'✅ Best DT params  : {best_dt_params}')
print(f'   Validation RMSE : {best_dt_rmse:.2f} MW')

In [ ]:
# ── Random Forest grid ────────────────────────────────────────────────────────
print('🔍 Tuning Random Forest...')
rf_param_grid = {
    'n_estimators'    : [50, 100, 200],
    'max_depth'       : [10, 20, None],
    'min_samples_split': [2, 10],
    'min_samples_leaf' : [1, 5],
    'max_features'     : ['sqrt', 'log2'],
}

best_rf_rmse = np.inf
best_rf_params = {}

for params in ParameterGrid(rf_param_grid):
    rf = RandomForestRegressor(random_state=SEED, n_jobs=-1, **params)
    rf.fit(X_tr, Y_tr)
    pred = rf.predict(X_va)
    rmse = np.sqrt(mean_squared_error(Y_va.flatten(), pred.flatten()))
    if rmse < best_rf_rmse:
        best_rf_rmse   = rmse
        best_rf_params = params

print(f'✅ Best RF params  : {best_rf_params}')
print(f'   Validation RMSE : {best_rf_rmse:.2f} MW')

## Step 7: Full Cross-Validation Training & Evaluation

In [ ]:
def run_cv(horizon, dt_params, rf_params):
    """
    Run 5-fold expanding window CV for both DT and RF at given horizon.
    Returns metrics DataFrame and fold predictions dict.
    """
    results = []
    fold_preds = {}   # {fold: {'actual': ..., 'dt_pred': ..., 'rf_pred': ..., 'index': ...}}

    for s in CV_SPLITS:
        fold = s['fold']
        train_data = df_feat.loc['2015-01-01' : s['train_end']]
        val_data   = df_feat.loc[s['val_start'] : s['val_end']]

        X_tr, Y_tr, _      = build_direct_multioutput_dataset(train_data, FEATURE_COLS, TARGET_COL, horizon)
        X_va, Y_va, va_idx = build_direct_multioutput_dataset(val_data,   FEATURE_COLS, TARGET_COL, horizon)

        # ── Decision Tree ──────────────────────────────────────────────────────
        dt_model = MultiOutputRegressor(
            DecisionTreeRegressor(random_state=SEED, **dt_params), n_jobs=-1
        )
        dt_model.fit(X_tr, Y_tr)
        dt_pred = dt_model.predict(X_va)
        dt_m    = compute_metrics(Y_va, dt_pred)

        # ── Random Forest ──────────────────────────────────────────────────────
        rf_model = RandomForestRegressor(random_state=SEED, n_jobs=-1, **rf_params)
        rf_model.fit(X_tr, Y_tr)
        rf_pred = rf_model.predict(X_va)
        rf_m    = compute_metrics(Y_va, rf_pred)

        results.append({
            'Fold': f'CV{fold}',
            'Horizon': f'{horizon}h',
            'Train Period': f"2015 → {s['train_end'][:4]}",
            'Val Period'  : f"{s['val_start'][:4]}",
            'DT_MAE' : dt_m['MAE'],  'DT_RMSE' : dt_m['RMSE'],  'DT_MAPE' : dt_m['MAPE'],
            'RF_MAE' : rf_m['MAE'],  'RF_RMSE' : rf_m['RMSE'],  'RF_MAPE' : rf_m['MAPE'],
        })

        fold_preds[fold] = {
            'actual' : Y_va,
            'dt_pred': dt_pred,
            'rf_pred': rf_pred,
            'index'  : va_idx,
            'dt_model': dt_model,
            'rf_model': rf_model,
        }

        print(f'  Fold {fold} | DT RMSE: {dt_m["RMSE"]:7.2f} MW  MAPE: {dt_m["MAPE"]:5.2f}%  |  '
              f'RF RMSE: {rf_m["RMSE"]:7.2f} MW  MAPE: {rf_m["MAPE"]:5.2f}%')

    return pd.DataFrame(results), fold_preds


# ── Run for both horizons ──────────────────────────────────────────────────────
print('=' * 80)
print(f'▶ Running CV — Horizon 12h')
print('=' * 80)
results_12, preds_12 = run_cv(HORIZON_12, best_dt_params, best_rf_params)

print()
print('=' * 80)
print(f'▶ Running CV — Horizon 24h')
print('=' * 80)
results_24, preds_24 = run_cv(HORIZON_24, best_dt_params, best_rf_params)

In [ ]:
# ── Summary tables ────────────────────────────────────────────────────────────
def print_summary(results_df, horizon):
    print(f'\n📊 Results Summary — {horizon}h Horizon')
    display(results_df[['Fold','Train Period','Val Period',
                         'DT_MAE','DT_RMSE','DT_MAPE',
                         'RF_MAE','RF_RMSE','RF_MAPE']].round(2))

    avg = results_df[['DT_MAE','DT_RMSE','DT_MAPE',
                       'RF_MAE','RF_RMSE','RF_MAPE']].mean()
    print(f'\n📈 Average across folds:')
    print(f'   Decision Tree  — MAE: {avg.DT_MAE:.2f} MW  |  RMSE: {avg.DT_RMSE:.2f} MW  |  MAPE: {avg.DT_MAPE:.2f}%')
    print(f'   Random Forest  — MAE: {avg.RF_MAE:.2f} MW  |  RMSE: {avg.RF_RMSE:.2f} MW  |  MAPE: {avg.RF_MAPE:.2f}%')

print_summary(results_12, 12)
print_summary(results_24, 24)

## Step 8: Visualisation — Actual vs Predicted

In [ ]:
def plot_predictions(fold_preds, horizon, n_folds=5, plot_days=14):
    """
    For each CV fold, plot the first `plot_days` days of actual vs predicted.
    Uses step-1-ahead values from the multi-output predictions for clarity.
    """
    fig, axes = plt.subplots(n_folds, 1, figsize=(16, 4 * n_folds))
    fig.suptitle(f'Actual vs Predicted — {horizon}h Horizon (Step-1-Ahead shown)',
                 fontsize=14, fontweight='bold', y=1.01)

    for fold in range(1, n_folds + 1):
        ax  = axes[fold - 1]
        fp  = fold_preds[fold]
        idx = fp['index']

        # Use the first step-ahead prediction for visual clarity
        actual  = fp['actual'][:, 0]
        dt_pred = fp['dt_pred'][:, 0]
        rf_pred = fp['rf_pred'][:, 0]

        n_pts = min(plot_days * 24, len(actual))

        ax.plot(idx[:n_pts],  actual[:n_pts],  lw=1.2, label='Actual',       color='steelblue', alpha=0.9)
        ax.plot(idx[:n_pts],  dt_pred[:n_pts], lw=1.0, label='Decision Tree', color='coral',     alpha=0.85, linestyle='--')
        ax.plot(idx[:n_pts],  rf_pred[:n_pts], lw=1.0, label='Random Forest', color='seagreen',  alpha=0.85, linestyle='-.')

        ax.set_title(f'CV{fold}', fontsize=11, fontweight='bold')
        ax.set_ylabel('Demand (MW)')
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

    plt.tight_layout()
    plt.show()


plot_predictions(preds_12, horizon=12)
plot_predictions(preds_24, horizon=24)

In [ ]:
# ── Scatter plot: actual vs predicted (best fold = CV4) ───────────────────────
def scatter_actual_pred(fold_preds, horizon, fold=4):
    fp      = fold_preds[fold]
    actual  = fp['actual'].flatten()
    dt_pred = fp['dt_pred'].flatten()
    rf_pred = fp['rf_pred'].flatten()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Scatter: Actual vs Predicted — {horizon}h Horizon (CV{fold})',
                 fontsize=13, fontweight='bold')

    for ax, pred, label, color in zip(
        axes,
        [dt_pred, rf_pred],
        ['Decision Tree', 'Random Forest'],
        ['coral', 'seagreen']
    ):
        ax.scatter(actual, pred, alpha=0.15, s=2, color=color)
        lim = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
        ax.plot(lim, lim, 'k--', lw=1.2, label='Perfect fit')
        ax.set_xlabel('Actual (MW)');  ax.set_ylabel('Predicted (MW)')
        ax.set_title(label)
        ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


scatter_actual_pred(preds_12, horizon=12)
scatter_actual_pred(preds_24, horizon=24)

## Step 9: Feature Importance Analysis

In [ ]:
def plot_feature_importance(fold_preds, horizon, fold=4, top_n=20):
    """
    Plot top-N feature importances for DT and RF from the selected fold.
    For MultiOutputRegressor DT, average across estimators.
    For RF (native multi-output), use built-in importances.
    """
    fp = fold_preds[fold]

    # Decision Tree: average importances across all output estimators
    dt_importances = np.mean(
        [est.feature_importances_ for est in fp['dt_model'].estimators_], axis=0
    )

    # Random Forest: native feature importances
    rf_importances = fp['rf_model'].feature_importances_

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Feature Importance — {horizon}h Horizon (CV{fold})',
                 fontsize=13, fontweight='bold')

    for ax, imp, label, color in zip(
        axes,
        [dt_importances, rf_importances],
        ['Decision Tree', 'Random Forest'],
        ['coral', 'seagreen']
    ):
        idx_sorted = np.argsort(imp)[-top_n:][::-1]
        top_feats  = [FEATURE_COLS[i] for i in idx_sorted]
        top_vals   = imp[idx_sorted]

        bars = ax.barh(range(top_n), top_vals[::-1], color=color, alpha=0.8, edgecolor='white')
        ax.set_yticks(range(top_n))
        ax.set_yticklabels(top_feats[::-1], fontsize=8)
        ax.set_title(label, fontsize=11, fontweight='bold')
        ax.set_xlabel('Importance')
        ax.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    plt.show()


plot_feature_importance(preds_12, horizon=12)
plot_feature_importance(preds_24, horizon=24)

## Step 10: Robustness & Overfitting Analysis

In [ ]:
# ── Train vs Validation RMSE across folds (overfitting check) ─────────────────
def compute_train_val_rmse(horizon, dt_params, rf_params):
    rows = []
    for s in CV_SPLITS:
        fold = s['fold']
        train_data = df_feat.loc['2015-01-01' : s['train_end']]
        val_data   = df_feat.loc[s['val_start'] : s['val_end']]

        X_tr, Y_tr, _ = build_direct_multioutput_dataset(train_data, FEATURE_COLS, TARGET_COL, horizon)
        X_va, Y_va, _ = build_direct_multioutput_dataset(val_data,   FEATURE_COLS, TARGET_COL, horizon)

        for model_name, model in [
            ('Decision Tree', MultiOutputRegressor(DecisionTreeRegressor(random_state=SEED, **dt_params), n_jobs=-1)),
            ('Random Forest', RandomForestRegressor(random_state=SEED, n_jobs=-1, **rf_params)),
        ]:
            model.fit(X_tr, Y_tr)
            tr_rmse = np.sqrt(mean_squared_error(Y_tr.flatten(), model.predict(X_tr).flatten()))
            va_rmse = np.sqrt(mean_squared_error(Y_va.flatten(), model.predict(X_va).flatten()))
            rows.append({'Fold': f'CV{fold}', 'Model': model_name,
                         'Train RMSE': tr_rmse, 'Val RMSE': va_rmse,
                         'Gap': va_rmse - tr_rmse})
    return pd.DataFrame(rows)


ov_12 = compute_train_val_rmse(HORIZON_12, best_dt_params, best_rf_params)
ov_24 = compute_train_val_rmse(HORIZON_24, best_dt_params, best_rf_params)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, ov, h in zip(axes, [ov_12, ov_24], [12, 24]):
    for model, color in [('Decision Tree', 'coral'), ('Random Forest', 'seagreen')]:
        sub = ov[ov['Model'] == model]
        ax.plot(sub['Fold'], sub['Train RMSE'], 'o--', color=color, alpha=0.7, label=f'{model} Train')
        ax.plot(sub['Fold'], sub['Val RMSE'],   's-',  color=color, alpha=1.0, label=f'{model} Val',
                linewidth=2)
    ax.set_title(f'Train vs Val RMSE — {h}h Horizon', fontsize=12, fontweight='bold')
    ax.set_ylabel('RMSE (MW)');  ax.set_xlabel('CV Fold')
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nOverfitting gap (Val RMSE − Train RMSE):')
display(ov_12.groupby('Model')[['Train RMSE','Val RMSE','Gap']].mean().round(2))

In [ ]:
# ── RMSE comparison bar chart: DT vs RF, 12h vs 24h ───────────────────────────
avg_12 = results_12[['DT_RMSE','RF_RMSE']].mean()
avg_24 = results_24[['DT_RMSE','RF_RMSE']].mean()

labels  = ['Decision Tree\n12h', 'Random Forest\n12h',
           'Decision Tree\n24h', 'Random Forest\n24h']
values  = [avg_12['DT_RMSE'], avg_12['RF_RMSE'],
           avg_24['DT_RMSE'], avg_24['RF_RMSE']]
colors  = ['#E57373','#4DB6AC','#EF9A9A','#80CBC4']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Average RMSE: Model × Horizon Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('RMSE (MW)');  ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ── Wilcoxon signed-rank test: DT vs RF ───────────────────────────────────────
print('📐 Wilcoxon Signed-Rank Test (DT vs RF per-fold RMSE)')
for h, res in [(12, results_12), (24, results_24)]:
    stat, p = wilcoxon(res['DT_RMSE'].values, res['RF_RMSE'].values)
    sig = '✅ Significant (p < 0.05)' if p < 0.05 else '❌ Not significant'
    print(f'  {h}h horizon  — stat={stat:.4f},  p={p:.4f}  →  {sig}')

## Step 11: Decision Tree Structure Visualisation

In [ ]:
# ── Visualise a shallow DT (max_depth=4) for interpretability ─────────────────
s = CV_SPLITS[3]  # CV4
train_data = df_feat.loc['2015-01-01' : s['train_end']]
X_vis, Y_vis, _ = build_direct_multioutput_dataset(train_data, FEATURE_COLS, TARGET_COL, HORIZON_12)

dt_vis = DecisionTreeRegressor(max_depth=4, random_state=SEED)
dt_vis.fit(X_vis, Y_vis[:, 0])  # single-output: step-1-ahead

fig, ax = plt.subplots(figsize=(28, 8))
plot_tree(
    dt_vis,
    feature_names=FEATURE_COLS,
    filled=True,
    rounded=True,
    fontsize=7,
    ax=ax,
    max_depth=3,
    impurity=False,
)
ax.set_title('Decision Tree Structure (depth ≤ 4, step-1-ahead target, CV4 training data)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 12: Final Results Summary

In [ ]:
print('=' * 70)
print('  FINAL RESULTS SUMMARY')
print('=' * 70)

for h, res in [(12, results_12), (24, results_24)]:
    avg = res[['DT_MAE','DT_RMSE','DT_MAPE','RF_MAE','RF_RMSE','RF_MAPE']].mean()
    print(f'\n  Horizon: {h}h')
    print(f'  {"Model":<20} {"MAE":>8} {"RMSE":>8} {"MAPE":>8}')
    print(f'  {"-"*46}')
    print(f'  {"Decision Tree":<20} {avg.DT_MAE:>8.2f} {avg.DT_RMSE:>8.2f} {avg.DT_MAPE:>7.2f}%')
    print(f'  {"Random Forest":<20} {avg.RF_MAE:>8.2f} {avg.RF_RMSE:>8.2f} {avg.RF_MAPE:>7.2f}%')

print('\n  Units: MAE / RMSE in MW;  MAPE in %')
print('=' * 70)